In [0]:
import dlt
from pyspark.sql.functions import *
import spark

@dlt.table(
  name="sales_silver",
  comment="Clean sales data"
)
def sales_silver():
    df = dlt.read("sales_bronze")

    return (
        df.filter(col("amount") > 0)
          .withColumn("amount", col("amount").cast("double"))
    )

In [0]:
import dlt
from pyspark.sql.functions import *

@dlt.table(
    name="analytics.silver.sales_silver",
    comment="Cleaned sales data"
)
def sales_silver():
    df = spark.read.table("dev.bronze.sales_bronze")

    return (
        df.filter(col("amount") > 0)
          .withColumn("amount", col("amount").cast("double"))
    )

### Silver Layer (Cleaning + Strong Data Quality)
Silver layer usually enforces strict data quality rules.

In [0]:
import dlt
from pyspark.sql.functions import *

# rules = {
#     "valid_amount": "amount > 0",
#     "valid_customer": "customer_id IS NOT NULL",
#     "valid_date": "order_date IS NOT NULL"
# }

@dlt.table(
    name="analytics.silver.sales_silver",
    comment="Cleaned sales data"
)
@dlt.expect_or_drop("valid_amount", "amount > 0")
@dlt.expect_or_drop("valid_customer", "customer_id IS NOT NULL")
# @dlt.expect_all_or_drop(rules)
def sales_silver():

    df = spark.read.table("dev.bronze.sales_bronze")

    return (
        df.withColumn("amount", col("amount").cast("double"))
          .withColumn("order_date", to_date("order_date"))
    )

### Silver Layer (Clean + SCD Type 2)
Silver performs data cleansing + deduplication + SCD2 merge logic.
DLT provides apply_changes for CDC/SCD patterns.

In [0]:
# Step 1 — Create the staging view
@dlt.view(
    name="sales_silver_staging"
)
def sales_silver_staging():
    return (
        dlt.read("dev.bronze.sales_bronze")
        .withColumn("updated_at", col("updated_at").cast("timestamp"))
        .dropDuplicates(["order_id", "updated_at"])
    )

In [0]:
# Step 2 — Create target Silver table
dlt.create_streaming_table(
    name="dev.silver.sales_silver",
    comment="Cleaned sales table with SCD Type 2 history"
)

In [0]:
# Step 3 — Apply SCD Type 2
dlt.apply_changes(
    target = "dev.silver.sales_silver",
    source = "sales_silver_staging",
    keys = ["order_id"],
    sequence_by = col("updated_at"),
    stored_as_scd_type = 2
)
# This automatically creates columns:
# __START_AT
# __END_AT
# __CURRENT
# and maintains historical versions.